# 02 · The repair-order state engine

**Purpose** — show the deterministic spine: an append-only event log folded into
current state, with illegal transitions rejected and contradictions surfaced.

**Prerequisites** — notebook 01 has been run. No GPU.

### Why event-sourced
State is **derived**, never overwritten. Two things fall out for free:
a complete audit trail, and citable provenance for every fact the agent states.

> **Compute deterministically, narrate with the LLM.** Every number here is
> computed in Python. The LLM only ever narrates these results.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## The lifecycle

In [ ]:
from app.state.transitions import LEGAL, ROState, BLOCKING_STATES
for src, dsts in LEGAL.items():
    arrow = ", ".join(sorted(d.value for d in dsts)) or "(terminal)"
    mark = "  <-- blocking" if src in BLOCKING_STATES else ""
    print(f"{src.value:24s} -> {arrow}{mark}")

## Execute — fold a real repair order

In [ ]:
from datetime import datetime
from app.state import db as dbm
from app.state.engine import fold
con = dbm.connect()
ro_no = con.execute(
    "SELECT ro_number FROM ros ORDER BY ro_number LIMIT 1").fetchone()[0]
ro = dbm.get_ro(con, ro_no)
events = dbm.events_for_ro(con, ro_no)
snap = fold(events, promised_time=datetime.fromisoformat(ro["promised_time"]))
print(f"{ro_no}: {len(events)} events -> state {snap.state.value}")
print("completed :", [(o.op_code, o.actual_hrs) for o in snap.completed_ops])
print("pending   :", [o.op_code for o in snap.pending_ops])
print("hours booked", snap.hours_booked, "| flat-rate earned", snap.flat_rate_total,
      "| proficiency", snap.proficiency)

## What you should see — illegal transitions are rejected, not applied

A repair cannot begin before the customer has authorised the work. The engine
refuses the move and records a conflict rather than silently accepting it.

In [ ]:
from app.state.events import Event, EventType as E
from datetime import timedelta
t0 = datetime(2026, 9, 24, 8, 0)
evs = [Event("RO-TEST", E.RO_OPENED, t0, "SYSTEM"),
       Event("RO-TEST", E.STATE_CHANGED, t0+timedelta(minutes=5), "ADV001", {"to": "DISPATCHED"}),
       Event("RO-TEST", E.STATE_CHANGED, t0+timedelta(minutes=10), "ADV001", {"to": "INVOICED"})]
s = fold(evs)
print("state after illegal jump:", s.state.value, "  (stayed put)")
for c in s.conflicts: print("conflict:", c)

## What you should see — an out-of-spec measurement raises a safety flag

In [ ]:
evs = [Event("RO-TEST", E.RO_OPENED, t0, "SYSTEM"),
       Event("RO-TEST", E.MEASUREMENT_TAKEN, t0+timedelta(minutes=30), "EMP014",
             {"type": "rotor_thickness", "value": 22.8, "unit": "mm",
              "spec_min": 23.0, "out_of_spec": True, "safety_related": True})]
s = fold(evs)
print("open safety item:", s.has_open_safety)
for f in s.safety_flags: print("  ", f["detail"], " (from event", f["event_id"], ")")

## Verify — the full test suite

In [ ]:
!.venv/bin/python -m pytest tests/ -q || python -m pytest tests/ -q